In [13]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)


python-dotenv could not parse statement starting at line 17


In [5]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    '''人物信息'''
    name:str = Field(description='姓名')
    age : int =  Field(description='年龄')
    job :str = Field(description='职业')

#创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)


result = structured_model.invoke('张三是一名30岁的软件工程师')

print(result)
print(type(result))

name='张三' age=30 job='软件工程师'
<class '__main__.Person'>


In [6]:
print(f'年龄:{result.age}')
print(f'姓名:{result.name}')
print(f'职业:{result.job}')

年龄:30
姓名:张三
职业:软件工程师


举例二

In [7]:
class MovieModel(BaseModel):
    '''电影的详细信息'''
    title : str = Field(description='电影标题')
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


structured_model = model.with_structured_output(MovieModel)
result = structured_model.invoke('给出电影盗梦空间的信息')

print(result)
print(type(model))

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=9.0
<class 'langchain_community.chat_models.zhipuai.ChatZhipuAI'>


举例三


In [8]:
from pydantic import BaseModel, Field
# 定义输出结构
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")

# ✅ v1.x：使用 with_structured_output
structured_model = model.with_structured_output(SentimentAnalysis)

# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
    f"分析以下文本的情感：\n{text}"
)

print(f"类型: {type(result)}")  # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.97
关键词: ['实用', '学到知识', '强烈推荐']


## 2、高级特性

### 2.1 高级字段
举例

In [22]:
from pydantic import BaseModel, Field
from typing import Optional



class Person(BaseModel):
    '''人物信息'''
    name:str = Field(description='姓名')
    age : Optional[int] =  Field(description='年龄')
    job :str = Field(description='职业')

#创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)


res = structured_model.invoke('张三是一名软件工程师')

print(res)
print(type(res))

name='张三' age=None job='软件工程师'
<class '__main__.Person'>


In [23]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    '''人物信息'''
    name:str = Field(description='姓名')
    age : int =  Field(description='年龄')
    job :str = Field(description='职业')

#创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)


result = structured_model.invoke('张三是一名软件工程师')

print(result)
print(type(result))

None
<class 'NoneType'>


### 列表提取

举例

In [24]:
from typing import List

class Person(BaseModel):
    """人物信息"""
    name: str
    age: int

class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person]  # 多个 Person 对象

structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke("张三 30岁，李四 25岁")

print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=25)]


举例2

In [25]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model.with_structured_output(Review)

review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")

print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


### 嵌套结构
举例

In [26]:
from pydantic import BaseModel

class Address(BaseModel):
    """地点描述"""
    city: str
    district: str

class Company(BaseModel):
    """公司信息"""
    name: str
    address: Address  # 嵌套模型

structured_llm = model.with_structured_output(Company)

result = structured_llm.invoke("阿里巴巴在杭州滨江区")

print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


举例2

In [27]:
from pydantic import BaseModel, Field
from typing import List

# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")  # 定义列表字段
    rating: float = Field(description="评分")

# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)

# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")

# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='多姆·柯布'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='艾伦·佩吉', role='阿丽雅德妮'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='渡边谦', role='斋藤'), Actor(name='玛丽昂·歌迪亚', role='玛尔'), Actor(name='基里安·墨菲', role='罗伯特·费舍尔'), Actor(name='迪利普·劳', role='优素福')]
评分: 9.3


## 限制条件

举例

In [28]:
from pydantic import ValidationError

class User(BaseModel):
    name: str = Field(min_length=2, max_length=20)
    age: int = Field(ge=0, le=150)
    email: str

print("\n有效数据:")
try:
    user = User(name="张三", age=30, email="zhang@example.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] {e}")

print("\n无效数据（年龄超出范围）:")
try:
    user = User(name="李四", age=200, email="li@example.com")
    print(f"[OK] {user}")
except ValidationError as e:
    print(f"[FAIL] 验证失败（符合预期）: {e.errors()[0]['msg']}")


有效数据:
[OK] 张三, 30, zhang@example.com

无效数据（年龄超出范围）:
[FAIL] 验证失败（符合预期）: Input should be less than or equal to 150
